# AAL Sales Analysis | Course-End Project

**Applied Data Science with Python, Q4 2020.** This is our working notebook, following rubric sections 1 → 2 → 3 → 4. We will complete Section 1 before starting the others. The CSV does not define a currency, so we do not assume AUD.

| Section | Status |
|---|---|
| 1. Data Wrangling | Ready to run |
| 2. Data Analysis | To be completed |
| 3. Data Visualization | To be completed |
| 4. Report Generation | To be completed |

## 1. Data Wrangling

### 1a. Inspect missing and incorrect entries

**Step 1:** Read the original CSV. Practice Labs keeps it next to your notebook; the project folder keeps the original ZIP in `data/`. Both work.

In [ ]:
from pathlib import Path
import pandas as pd
candidates = [Path('AusApparalSales4thQrt2020.csv'), Path('data/AusApparalSales4thQrt2020.csv'),
              Path('../data/AusApparalSales4thQrt2020.csv'), Path('1739523823_ausapparalsales4thqrt2020.zip'),
              Path('data/1739523823_ausapparalsales4thqrt2020.zip'), Path('../data/1739523823_ausapparalsales4thqrt2020.zip')]
source_file = next((p for p in candidates if p.is_file()), None)
if source_file is None:
    raise FileNotFoundError('Place the CSV beside the notebook or the original ZIP in data/.')
raw_df = pd.read_csv(source_file)
print('Source:', source_file)
print(f'Rows: {len(raw_df):,}; columns: {len(raw_df.columns)}')
display(raw_df.head())

**Step 2:** Examine data types and missing values using both `isna()` and `notna()`.



The six original columns are `Date`, `Time`, `State`, `Group`, `Unit` and `Sales`. Notice that the original categorical fields contain spaces after commas. We can trim those consistently rather than pretending they are different labels.

In [ ]:
print(raw_df.dtypes)
print('Raw nulls per column:')
display(raw_df.isna().sum().to_frame('isna() count'))
print('Raw non-nulls per column:')
display(raw_df.notna().sum().to_frame('notna() count'))

**Step 3:** Work on a copy. Clean whitespace and parse dates/numbers explicitly.



Dates are parsed with an explicit day-month-year format. Non-numeric or invalid entries become null for inspection, never silently converted to a guessed value. The business-key duplicate check uses one record per date × time × state × customer group.

In [ ]:
df = raw_df.copy()
for column in ['Time', 'State', 'Group']:
    df[column] = df[column].astype('string').str.strip()
df['Date'] = pd.to_datetime(df['Date'].astype('string').str.strip(), format='%d-%b-%Y', errors='coerce')
for column in ['Unit', 'Sales']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

quality = pd.DataFrame({
    'Missing or invalid': df.isna().sum(),
    'Present': df.notna().sum(),
})
display(quality)

key_cols = ['Date', 'Time', 'State', 'Group']
checks = pd.Series({
    'Fully identical duplicate rows': int(df.duplicated().sum()),
    'Duplicate business keys': int(df.duplicated(subset=key_cols).sum()),
    'Non-positive units': int((df['Unit'] <= 0).sum()),
    'Negative sales values': int((df['Sales'] < 0).sum()),
    'Non-integer unit values': int(((df['Unit'] % 1 != 0) & df['Unit'].notna()).sum()),
})
display(checks.to_frame('Count'))

**Step 4:** Check for absent whole dates. Empty cells and completely missing dates are different problems.



A table can have no missing *cells* and still omit entire dates. This is a different form of missing data. We compare observed dates against every calendar day of Q4 2020 rather than just the earliest and latest recorded dates.

In [ ]:
quarter_days = pd.date_range('2020-10-01', '2020-12-31', freq='D')
observed_days = pd.DatetimeIndex(df['Date'].dropna().unique())
missing_days = quarter_days.difference(observed_days)
print('Observed days:', len(observed_days), 'out of', len(quarter_days))
print('Missing dates:', [d.strftime('%d %b %Y') for d in missing_days])
print('Rows per observed day (minimum, maximum):', df.groupby('Date').size().agg(['min', 'max']).to_dict())
print('Rows outside Q4 2020:', int((~df['Date'].between(quarter_days.min(), quarter_days.max())).sum()))

### 1b. Recommend how to treat missing or incorrect data

- **Null values:** The source has zero missing cells, so we should not fill or drop any existing rows.
- **Formatting:** Strip whitespace so category names are consistent.
- **Suspected incorrect values or duplicates:** Investigate the source before deleting or imputing.
- **Missing whole days (31 October and 31 December 2020):** Keep the dates flagged as unrecorded. Do not invent zero sales or median sales; `dropna()` and `fillna()` do not restore absent rows.

**Holiday hypothesis (unverified):** 31 October is Halloween; 31 December is New Year's Eve. Altered trading or closures are possible, but a reporting/extraction gap is also plausible. No store-opening calendar is supplied. Halloween is not generally an Australian retail closure day, so neither date can be called a confirmed closure.

### 1c. Choose and perform normalization

**Min-Max normalization** is preferred by the course brief. The transformation `(x - min(x)) / (max(x) - min(x))` places numeric values between **0 and 1**. We normalize `Unit` and `Sales` into new columns; the original values stay untouched.

In [ ]:
for column in ['Unit', 'Sales']:
    minimum = df[column].min()
    maximum = df[column].max()
    if minimum == maximum:
        raise ValueError(f'Cannot normalize {column}: constant values cause division by zero')
    df[column + '_normalized'] = (df[column] - minimum) / (maximum - minimum)

print('Original and normalized values:')
display(df[['Unit', 'Unit_normalized', 'Sales', 'Sales_normalized']].head(8))
print('Normalization bounds:')
display(df[['Unit_normalized', 'Sales_normalized']].agg(['min', 'max']))

**Check for redundant information:** In this particular dataset, `Sales` equals `Unit × 2500`, so their separately normalized columns should be numerically identical.

In [ ]:
price_per_unit = df['Sales'] / df['Unit']
print('Unique sales/unit ratios:', sorted(price_per_unit.dropna().unique().tolist()))
print('Rows that do not match Unit × 2500:', int((df['Sales'] != df['Unit'] * 2500).sum()))
display(df[['Unit', 'Sales']].describe().round(2))
print('Maximum normalized-column difference:', (df['Unit_normalized'] - df['Sales_normalized']).abs().max())

### 1d. Demonstrate `GroupBy()` and recommend its use

`groupby()` **splits** rows into logical chunks, **applies** an aggregation such as `sum()` or `size()`, then **combines** the summaries. Grouping (chunking) is appropriate for state/demographic summaries. **Merging** would join a second dataset such as store locations, which we do not have. Sum original sales and units, not normalized values.

In [ ]:
by_state = (df.groupby('State', as_index=False)
    .agg(Records=('Sales','size'), Total_Units=('Unit','sum'), Total_Sales=('Sales','sum'))
    .sort_values('Total_Sales', ascending=False))
by_state_and_group = (df.groupby(['State','Group'], as_index=False)
    .agg(Records=('Sales','size'), Total_Sales=('Sales','sum'))
    .sort_values(['State','Group']))
print('State summary:')
display(by_state)
print('State × demographic group (first ten rows):')
display(by_state_and_group.head(10))

**Verification for Section 1.** These checks should pass before moving on.

In [ ]:
assert len(df) == len(raw_df)
assert df[['Unit_normalized','Sales_normalized']].notna().all().all()
assert df['Unit_normalized'].between(0,1).all()
assert df['Sales_normalized'].between(0,1).all()
assert abs(by_state['Total_Sales'].sum()-df['Sales'].sum()) < 1e-6
assert (df['Sales_normalized'] - df['Unit_normalized']).abs().max() < 1e-9
print('Section 1 checks passed: rows preserved, normalization within bounds, group totals reconciled.')

## 2. Data Analysis

**To be completed next.** We will calculate mean, median, mode and standard deviation for Sales and Unit; highest/lowest demographic and state totals; weekly, monthly and quarterly reports.

## 3. Data Visualization

**To be completed after Section 2.** We will chart state × group sales, demographic group × state sales, time of day, plus daily/weekly/monthly/quarterly patterns, and explain our choice of Seaborn.

## 4. Report Generation

**To be completed last.** We will combine JupyterLab Markdown, box plots, Seaborn distributions, verified findings, caveats and business recommendations.